# 09 · Exportar resultados a LaTeX

Lee las tablas de las fases 1 y 2 y genera un documento LaTeX **liviano** con las fórmulas, los resultados significativos y sus interpretaciones. Sirve para compartir un PDF (o subirlo a Overleaf) sin los notebooks, que pesan mucho por sus salidas.

| | |
|---|---|
| **Entrada** | `resultados/tablas/fase1_consolidada.csv`, `resultados/tablas/fase2_cruzados.csv` y `resultados/figuras/figura_matriz_recomendacion.png` |
| **Salida** | `resultados/latex/resultados_modelos.tex` (+ la figura) y, si hay `pdflatex`, `resultados_modelos.pdf` |

**Para Overleaf:** subir `resultados_modelos.tex` y `figura_matriz_recomendacion.png` y compilar con pdfLaTeX. Las interpretaciones se generan con reglas sobre los números: léanlas y ajústenlas con sus palabras.

In [ ]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook genera un archivo LaTeX con las fórmulas, los resultados significativos y sus interpretaciones.
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os
import shutil
import subprocess

import numpy as np
import pandas as pd

## 1. Parámetros y funciones auxiliares

In [ ]:
# Parámetros
# ------------------------------------------------------------------------------------------------------------
UMBRAL_P = 0.05             # significancia de la fase 1 (sin corrección)
MIN_SKUS_CONFIABLE = 100    # por debajo de este número de SKUs se advierte muestra pequeña

NOMBRES_VARIABLES = {
    "onpromotion": "promoción propia (día)",
    "onpromotion_l1_3": "promoción propia (rezago 1-3 días)",
    "onpromotion_l4_6": "promoción propia (rezago 4-6 días)",
    "efecto_neto_propio": "efecto neto propio",
    "promo_clase_otros": "otros SKUs de la clase (día)",
    "promo_clase_otros_l1_3": "otros SKUs de la clase (rezago 1-3)",
    "promo_clase_otros_l4_6": "otros SKUs de la clase (rezago 4-6)",
    "promo_cat_otras_clases": "resto de la categoría (día)",
    "promo_cat_otras_clases_l1_3": "resto de la categoría (rezago 1-3)",
    "promo_cat_otras_clases_l4_6": "resto de la categoría (rezago 4-6)",
}
PROPIAS = ["onpromotion", "onpromotion_l1_3", "onpromotion_l4_6", "efecto_neto_propio"]

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_tablas = os.path.join("resultados", "tablas")
ruta_figuras = os.path.join("resultados", "figuras")
ruta_salida = os.path.join("resultados", "latex")
compilar_pdf = True      # True: compila el PDF si hay pdflatex instalado

In [ ]:
def tex(texto: str) -> str:
    """
    Esta función escapa los caracteres especiales de LaTeX en un texto.

    :param texto:   el texto original.
    :return:        el texto listo para insertar en LaTeX.
    """

    for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"), ("$", r"\$")]:
        texto = texto.replace(a, b)
    return texto


def num(x: float, dec: int = 1, signo: bool = True) -> str:
    """
    Esta función formatea un número con coma decimal.

    :param x:       el número.
    :param dec:     los decimales.
    :param signo:   si es True, antepone + a los positivos.
    :return:        el número como texto.
    """

    s = f"{x:+.{dec}f}" if signo else f"{x:.{dec}f}"
    return s.replace(".", ",")


def pct(x: float, dec: int = 1) -> str:
    """
    Esta función formatea un porcentaje con signo y coma decimal para LaTeX.

    :param x:       el porcentaje.
    :param dec:     los decimales.
    :return:        el texto, por ejemplo '+59,8\\%'.
    """

    return num(x, dec) + r"\%"


def estrellas(p: float) -> str:
    """
    Esta función devuelve las estrellas de significancia de un valor p.

    :param p:   el valor p.
    :return:    '***' si p < 0,001; '**' si p < 0,01; '*' si p < 0,05; vacío en otro caso.
    """

    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


def p_texto(p: float) -> str:
    """
    Esta función formatea un valor p.

    :param p:   el valor p.
    :return:    '<0,001' o el valor con tres decimales.
    """

    return "$<$0,001" if p < 0.001 else num(p, 3, signo=False)


def efecto_10pp(coef: float) -> float:
    """
    Esta función convierte un coeficiente de proporción (0 a 1) en el efecto porcentual de +10 puntos porcentuales.

    :param coef:    el coeficiente del PPML.
    :return:        100 (exp(0,1 coef) - 1).
    """

    return 100 * (np.exp(0.1 * coef) - 1)

## 2. Textos fijos del documento (fórmulas y limitaciones)

In [ ]:
ENCABEZADO = r"""\documentclass[11pt,a4paper]{article}
\usepackage[T1]{fontenc}
\usepackage[utf8]{inputenc}
\usepackage[spanish]{babel}
\usepackage{lmodern}
\usepackage[margin=2.3cm]{geometry}
\usepackage{amsmath,amssymb}
\usepackage{booktabs}
\usepackage{graphicx}
\usepackage{array}
\usepackage{float}
\usepackage[hidelinks]{hyperref}
\title{Resultados de la estimaci\'on de los modelos\\[4pt]\large Efectividad de las promociones: impacto y canibalizaci\'on en las ventas del retail}
\author{}
\date{Octubre de 2026}
\begin{document}
\maketitle
"""

ESPECIFICACION = r"""
\section{Especificaci\'on de los modelos}

Se estima un modelo por categor\'{\i}a objetivo $j$ con regresi\'on de Poisson por pseudo m\'axima
verosimilitud (PPML) con efectos fijos, sobre las ventas diarias por SKU de la tienda 44 (Quito). Un coeficiente
$\beta$ implica que las ventas cambian en $100\,(e^{\beta}-1)$\,\% cuando la variable aumenta en una unidad.

\paragraph{Fase 1: efecto propio y dentro de la categor\'{\i}a.}
\begin{equation}
\mathbb{E}[y_{it}] = \exp\Big(
  \alpha_i + \gamma_{\text{dow}(t)} + \tau_{m(t)} + \mathbf{h}_t'\boldsymbol{\theta}
  + \sum_{b\in\mathcal{B}} \beta_b\, D^{(b)}_{it}
  + \sum_{b\in\mathcal{B}} \psi^{\text{cl}}_b\, S^{\text{cl}(b)}_{it}
  + \sum_{b\in\mathcal{B}} \psi^{\text{cat}}_b\, S^{\text{cat}(b)}_{it}
\Big)
\end{equation}
donde $y_{it}$ son las ventas del SKU $i$ el d\'{\i}a $t$; $\alpha_i$, $\gamma_{\text{dow}(t)}$ y $\tau_{m(t)}$ son
efectos fijos de SKU, d\'{\i}a de la semana y mes; $\mathbf{h}_t$ son los indicadores de calendario; $D_{it}$ es la
promoci\'on del propio SKU; $S^{\text{cl}}_{it}$ y $S^{\text{cat}}_{it}$ son la proporci\'on de los otros SKUs de su
clase y de las otras clases de su categor\'{\i}a en promoci\'on; y $\mathcal{B}=\{0,\ \ell_{1\text{-}3},\ \ell_{4\text{-}6}\}$
son el d\'{\i}a actual y dos bloques de rezagos. El efecto neto de la promoci\'on propia es
$e^{\beta_0+\beta_{\ell_{1\text{-}3}}+\beta_{\ell_{4\text{-}6}}}-1$.

\paragraph{Fase 2: efectos entre categor\'{\i}as.} A la ecuaci\'on anterior se agrega la intensidad promocional
$P_{kt}$ de las categor\'{\i}as $k\neq j$ del n\'ucleo y una variable de control $R_{jt}$ (resto):
\begin{equation}
\cdots + \sum_{k\in\mathcal{K}\setminus\{j\}}\ \sum_{b\in\mathcal{B}} \beta_{jk,b}\, P^{(b)}_{kt}
       + \sum_{b\in\mathcal{B}} \rho_b\, R^{(b)}_{jt},
\qquad
L_{jk} = e^{0{,}1\,\beta_{jk,0}} - 1 .
\end{equation}
$L_{jk}$ es el efecto sobre las ventas de $j$ de que 10 puntos porcentuales adicionales del surtido de $k$ est\'en en
promoci\'on. El coeficiente del resto $R_{jt}$ es solo un control y no se interpreta.

\paragraph{Inferencia.} Los errores est\'andar se agrupan por SKU y por semana. En la fase 2 se aplica la correcci\'on
de Benjamini y Hochberg (tasa de falsos descubrimientos de 5\,\%) sobre todos los efectos cruzados contempor\'aneos.
En la fase 1 se se\~nala la significancia sin correcci\'on (* $p<0{,}05$, ** $p<0{,}01$, *** $p<0{,}001$).
"""

LIMITACIONES = r"""
\section{Limitaciones}
\begin{itemize}
  \item Los resultados son efectos \emph{asociados}, no causales: la cadena decide cu\'ando promocionar y los controles
        de calendario reducen ese sesgo, pero no lo eliminan.
  \item Los programas semanales fijos (carnes, aves, mariscos, huevos y fiambres los viernes; frutas y verduras los
        mi\'ercoles) no se pueden estimar con estos datos; no deben interpretarse como ``sin efecto''.
  \item ``Sin efecto detectable'' no significa efecto nulo: con 544 d\'{\i}as como m\'aximo, la informaci\'on para los
        efectos entre categor\'{\i}as es limitada.
  \item Es una sola tienda, sin precios ni m\'argenes, y se supone que ventas igual a demanda (sin quiebres de stock).
  \item El efecto de los dem\'as SKUs se resume por clase y categor\'{\i}a: se supone que, dentro de cada grupo, da lo mismo
        cu\'al SKU est\'a en promoci\'on.
  \item No se realiz\'o an\'alisis de robustez ni se estim\'o el modelo de tr\'afico (fase 3).
\end{itemize}
"""

## 3. Secciones de resultados (generadas desde las tablas)

In [ ]:
def seccion_muestra(f1: pd.DataFrame) -> str:
    """
    Esta función arma la sección con el tamaño de la muestra de cada categoría.

    :param f1:  la tabla consolidada de la fase 1.
    :return:    el texto LaTeX de la sección.
    """

    m = f1.groupby("categoria")[["n_obs", "n_skus", "n_dias"]].first()
    miles = lambda x: f"{int(x):,}".replace(",", ".")
    filas = "\n".join(
        f"{tex(c)} & {miles(r.n_obs)} & {miles(r.n_skus)} & {int(r.n_dias)} \\\\"
        for c, r in m.iterrows()
    )
    return rf"""
\section{{Muestra de estimaci\'on}}
\begin{{table}}[H]
\centering
\small
\begin{{tabular}}{{lrrr}}
\toprule
Categor\'{{\i}}a & Observaciones SKU-d\'{{\i}}a & SKUs & D\'{{\i}}as \\
\midrule
{filas}
\bottomrule
\end{{tabular}}
\caption{{Tama\~no de la muestra de cada modelo (ventana del terremoto y observaciones con rezagos incompletos excluidas).}}
\end{{table}}
"""


def seccion_fase1(f1: pd.DataFrame) -> str:
    """
    Esta función arma la sección de resultados de la fase 1 con tablas, solo los efectos significativos y su
    interpretación en lenguaje claro.

    :param f1:  la tabla consolidada de la fase 1.
    :return:    el texto LaTeX de la sección.
    """

    categorias = list(f1["categoria"].unique())

    # Tabla 1: efecto propio por categoría (efecto en porcentaje con estrellas)
    filas = []
    for c in categorias:
        t = f1.loc[f1["categoria"] == c].set_index("variable")
        celdas = [f"{pct(t.loc[v, 'efecto_pct'])}{estrellas(t.loc[v, 'p_valor'])}" for v in PROPIAS]
        filas.append(f"{tex(c)} & " + " & ".join(celdas) + r" \\")
    tabla_propio = "\n".join(filas)

    # Tabla 2: efectos significativos dentro de la categoría (clase y resto), por +10 puntos porcentuales
    sig = f1.loc[(~f1["variable"].isin(PROPIAS)) & (f1["p_valor"] < UMBRAL_P)].copy()
    sig["e10"] = sig["coef"].map(efecto_10pp)
    filas2 = "\n".join(
        f"{tex(r.categoria)} & {NOMBRES_VARIABLES[r.variable]} & {pct(r.e10)} & {p_texto(r.p_valor)} \\\\"
        for r in sig.itertuples()
    ) or r"\multicolumn{4}{c}{Ning\'un efecto significativo} \\"

    # Interpretaciones automáticas, solo con lo significativo
    texto = []
    for c in categorias:
        t = f1.loc[f1["categoria"] == c].set_index("variable")
        dia, neto = t.loc["onpromotion"], t.loc["efecto_neto_propio"]
        if dia["p_valor"] < UMBRAL_P:
            frase = (f"En {tex(c)}, un SKU en promoci\\'on se asocia con ventas {pct(dia['efecto_pct'])} "
                     f"respecto de un d\\'{{\\i}}a sin promoci\\'on (IC 95\\,\\%: {pct(dia['ic95_inf_pct'])} a "
                     f"{pct(dia['ic95_sup_pct'])}); el efecto neto, sumando los rezagos, es {pct(neto['efecto_pct'])}.")
        else:
            frase = f"En {tex(c)} no se detecta un efecto propio significativo el d\\'{{\\i}}a de la promoci\\'on."
        if t.loc["onpromotion_l1_3", "n_skus"] < MIN_SKUS_CONFIABLE:
            frase += f" La muestra es peque\\~na ({int(t.loc['onpromotion', 'n_skus'])} SKUs): interpretar con cautela."
        texto.append(r"\item " + frase)

    # Lectura conjunta de los rezagos
    l13 = f1.loc[f1["variable"] == "onpromotion_l1_3"]
    pos = int(((l13["coef"] > 0) & (l13["p_valor"] < UMBRAL_P)).sum())
    neg = int(((l13["coef"] < 0) & (l13["p_valor"] < UMBRAL_P)).sum())
    if pos and not neg:
        texto.append(rf"\item El rezago de 1 a 3 d\'{{\i}}as es positivo y significativo en {pos} de {len(l13)} categor\'{{\i}}as: "
                     r"no se observa una ca\'{\i}da posterior de las ventas (desplazamiento intertemporal), "
                     r"lo que contradice lo esperado en la metodolog\'{\i}a.")
    elif neg:
        texto.append(rf"\item El rezago de 1 a 3 d\'{{\i}}as es negativo y significativo en {neg} categor\'{{\i}}as, "
                     r"consistente con un desplazamiento intertemporal de las compras.")

    # Canibalización cercana (otros SKUs de la clase, día actual)
    cl = f1.loc[(f1["variable"] == "promo_clase_otros") & (f1["p_valor"] < UMBRAL_P)]
    neg_c = [tex(c) for c in cl.loc[cl["coef"] < 0, "categoria"]]
    pos_c = [tex(c) for c in cl.loc[cl["coef"] > 0, "categoria"]]
    if neg_c:
        texto.append(r"\item Cuando otros SKUs de su misma clase est\'an en promoci\'on, el SKU vende menos en: "
                     + ", ".join(neg_c) + r". Es compatible con canibalizaci\'on cercana.")
    if pos_c:
        texto.append(r"\item En " + ", ".join(pos_c) + r" el efecto de los otros SKUs de la clase es positivo, "
                     r"contrario a la canibalizaci\'on; conviene revisarlo antes de interpretarlo.")
    items = "\n".join(texto)

    return rf"""
\section{{Fase 1: efecto propio y dentro de la categor\'{{\i}}a}}

\begin{{table}}[H]
\centering
\small
\begin{{tabular}}{{lrrrr}}
\toprule
Categor\'{{\i}}a & D\'{{\i}}a & Rezago 1-3 & Rezago 4-6 & Efecto neto \\
\midrule
{tabla_propio}
\bottomrule
\end{{tabular}}
\caption{{Efecto asociado de la promoci\'on propia sobre las ventas del SKU (\%). * $p<0{{,}}05$, ** $p<0{{,}}01$, *** $p<0{{,}}001$.}}
\end{{table}}

\begin{{table}}[H]
\centering
\small
\begin{{tabular}}{{llrr}}
\toprule
Categor\'{{\i}}a & Variable & Efecto de +10 pp & Valor $p$ \\
\midrule
{filas2}
\bottomrule
\end{{tabular}}
\caption{{Efectos significativos de la promoci\'on de otros SKUs de la clase y del resto de la categor\'{{\i}}a
(cambio en las ventas cuando 10 puntos porcentuales adicionales del grupo est\'an en promoci\'on; $p<0{{,}}05$ sin correcci\'on).}}
\end{{table}}

\subsection*{{Interpretaci\'on}}
\begin{{itemize}}
{items}
\end{{itemize}}
"""


def seccion_fase2(f2: pd.DataFrame, hay_figura: bool) -> str:
    """
    Esta función arma la sección de la fase 2 con los pares significativos tras la corrección de Benjamini-Hochberg,
    la matriz de efectos y su interpretación.

    :param f2:          la tabla de efectos cruzados de la fase 2.
    :param hay_figura:  si la figura de la matriz está disponible.
    :return:            el texto LaTeX de la sección.
    """

    n = len(f2)
    sig = f2.loc[f2["significativo_bh"]].sort_values("p_bh")
    canibal = "canibalizaci\\'on"
    filas = "\n".join(
        f"{tex(r.afectada_j)} & {tex(r.en_promocion_k)} & {pct(r.L_10pp_pct)} & "
        f"[{num(r.L_ic95_inf_pct)}; {num(r.L_ic95_sup_pct)}] & {p_texto(r.p_bh)} & "
        + ("sinergia" if r.clasificacion == "sinergia" else canibal) + " \\\\"
        for r in sig.itertuples()
    ) or r"\multicolumn{6}{c}{Ning\'un par significativo} \\"

    frases = []
    for r in sig.itertuples():
        verbo = "aumentan" if r.L_10pp_pct > 0 else "disminuyen"
        tipo = "sinergia" if r.L_10pp_pct > 0 else "canibalizaci\\'on"
        frases.append(
            rf"\item Si 10 puntos porcentuales adicionales del surtido de {tex(r.en_promocion_k)} est\'an en promoci\'on, "
            rf"las ventas de {tex(r.afectada_j)} {verbo} {num(abs(r.L_10pp_pct), signo=False)}\,\% ({tipo}; "
            rf"valor $p$ corregido {p_texto(r.p_bh)})."
        )
    n_sin = int((~f2["significativo_bh"]).sum())
    frases.append(
        rf"\item Los otros {n_sin} de {n} pares son ``sin efecto detectable'' tras la correcci\'on: la evidencia no permite "
        r"distinguirlos de cero, lo que no equivale a un efecto nulo."
    )
    cats_sig = sorted(set(tex(c) for c in sig["afectada_j"]))
    if cats_sig:
        frases.append(r"\item Todos los pares significativos afectan a: " + ", ".join(cats_sig) +
                      r". Si esa categor\'{\i}a tiene pocos SKUs, los resultados deben leerse con cautela.")
    items = "\n".join(frases)

    figura = (r"""
\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{figura_matriz_recomendacion.png}
\caption{Matriz de efectos entre categor\'{\i}as. Filas: categor\'{\i}a afectada; columnas: categor\'{\i}a en promoci\'on.
* y recuadro: significativo tras Benjamini-Hochberg (FDR 5\,\%).}
\end{figure}
""" if hay_figura else "")

    return rf"""
\section{{Fase 2: efectos entre categor\'{{\i}}as}}

\begin{{table}}[H]
\centering
\small
\begin{{tabular}}{{llrrrl}}
\toprule
Afectada & En promoci\'on & Efecto de +10 pp & IC 95\,\% & $p$ (BH) & Clasificaci\'on \\
\midrule
{filas}
\bottomrule
\end{{tabular}}
\caption{{Efectos cruzados significativos tras la correcci\'on de Benjamini-Hochberg (FDR 5\,\%), de un total de {n} pares estimados.}}
\end{{table}}
{figura}
\subsection*{{Interpretaci\'on}}
\begin{{itemize}}
{items}
\end{{itemize}}
"""

## 4. Generar el documento

In [ ]:
# Leer las tablas de resultados
f1 = pd.read_csv(os.path.join(ruta_tablas, "fase1_consolidada.csv"))
f2 = pd.read_csv(os.path.join(ruta_tablas, "fase2_cruzados.csv"))
assert not f1.empty and not f2.empty, "las tablas de resultados están vacías"

# Copiar la figura de la matriz a la carpeta de salida
os.makedirs(ruta_salida, exist_ok=True)
origen_fig = os.path.join(ruta_figuras, "figura_matriz_recomendacion.png")
hay_figura = os.path.exists(origen_fig)
if hay_figura:
    shutil.copy(origen_fig, os.path.join(ruta_salida, "figura_matriz_recomendacion.png"))
else:
    print("Aviso: no se encontró la figura de la matriz; ejecutar 08_matriz_recomendacion.")

# Armar y guardar el documento
documento = (ENCABEZADO + ESPECIFICACION + seccion_muestra(f1) + seccion_fase1(f1)
             + seccion_fase2(f2, hay_figura) + LIMITACIONES + "\n\\end{document}\n")
archivo_tex = os.path.join(ruta_salida, "resultados_modelos.tex")
with open(archivo_tex, "w", encoding="utf-8") as f:
    f.write(documento)
print(f"Archivo LaTeX generado: {archivo_tex}")

# Compilar el PDF (opcional)
if compilar_pdf and shutil.which("pdflatex"):
    for _ in range(2):
        resultado = subprocess.run(["pdflatex", "-interaction=nonstopmode", "resultados_modelos.tex"],
                                   cwd=ruta_salida, capture_output=True, text=True)
    print("PDF generado." if resultado.returncode == 0 else "No se pudo compilar; subir el .tex a Overleaf.")
elif compilar_pdf:
    print("pdflatex no está instalado: subir el .tex y la figura a Overleaf para obtener el PDF.")

print(f"\n>>>> Proceso finalizado <<<<")